In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv("./data/base_clean.csv")

/var/folders/3_/3x44mfnn68xgbrckqv0zv5zc0000gn/T/ipykernel_42464/1050980111.py:1: DtypeWarning: Columns (20: first_chart_week, 23: charted_before_2000, 24: still_charting) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./data/base_clean.csv")


In [3]:
lyr = df[["id", "lyrics", "language_ft"]]

# Cleaning

Strip [] in lyrics

In [4]:
lyr["lyrics"] = (
    df["lyrics"]
    .str.replace(r"\[.*?\]", "", regex=True, flags=re.DOTALL)
    .str.replace(r"\n{2,}", "\n", regex=True)
    .str.strip()
)

Keep only "language_ft" = "en"

In [5]:
lyr = lyr[lyr["language_ft"]=="en"]

# Vocabulary

In [6]:
clean = (
    df["lyrics"]
    .str.lower()
    .str.replace(r"[^\w\s']", "", regex=True)
)

Word count

In [7]:
lyr["word_count"] = lyr["lyrics"].str.split().str.len()
lyr["unique_count"] = df["lyrics"].str.lower().str.split().apply(lambda w: len(set(w)) if  isinstance(w, list) else 0)

## Type-Token Ratio (TTR): unique words / total words

In [8]:
lyr["TTR"] = lyr["unique_count"] / lyr["word_count"]


Moving average TTR: Slide a window across the lyrics, computes TTR per window, and averages.

In [9]:
def mattr(text, window=50):
    if not isinstance(text, str):
        return float("nan")
    words = text.split()
    if len(words) < window:
        return float("nan")

    ttrs = [len(set(words[i:i+window])) / window for i in range(len(words) - window +1)]
    return sum(ttrs) / len(ttrs)

## Yule's K

K = 10000 * (\sum(f^2(i)) - word_count) / word_count^2

lower k_value means more diverse vocab and vice versa

In [10]:
from collections import Counter

def yules_k(text):
    if not isinstance(text, str):
        return float("nan")

    words = text.split()
    N = len(words)
    if N == 0:
        return float("nan")
    freqs = Counter(words).values()
    s2 = sum(f * f for f in freqs)
    return 10000 * (s2 - N) / (N**2)

lyr["yules_k"] = clean.apply(yules_k)

## Profanity

In [11]:
from better_profanity import profanity
from pandarallel import pandarallel

pandarallel.initialize()

profanity.load_censor_words()
PROFANE = frozenset(str(w).lower() for w in profanity.CENSOR_WORDSET)

def count_profanity(text):
    if not isinstance(text, str):
        return float("nan")
    return sum(w in PROFANE for w in text.split())

lyr["profanity_count"] = clean.parallel_apply(count_profanity)

INFO: Pandarallel will run on 10 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [12]:
lyr

,id,lyrics,language_ft,word_count,unique_count,TTR,yules_k,profanity_count
0,5A6OHHy73AR5tLxgTc98zz,"Yeah, uh-huh, you know what it is\nBlack and y...",en,656,200,0.304878,260.362509,8.0
1,0O45fw2L5vsWpdsOdXwNAR,I'm bringin' sexy back (Yeah)\nThem other boys...,en,673,126,0.187221,347.590016,4.0
2,2zZJnAlhgvW6DERn1tu3fb,"Ooh\nOoh, mmm, mmm\nHave you ever reached a ra...",en,389,156,0.401028,183.083439,1.0
3,0sf12qNH5qcw8qpgymFOqD,Yeah\nI've been tryna call\nI've been on my ow...,en,259,115,0.444015,178.646805,0.0
4,7wZUrN8oemZfsEd1CGkbXE,"Closed off from love, I didn't need the pain\n...",en,379,126,0.332454,291.468528,0.0
...,...,...,...,...,...,...,...,...
160195,1fXmDeiCb3ABt5CzkMxp4u,Yeah\nI saw you movin' on the dance floor\nI c...,en,371,148,0.398922,128.826615,0.0
160196,6sHxpnCerm9M6A5zaFFNfs,I'm sitting here cold as ice\nSwallow my frien...,en,320,118,0.368750,138.970719,3.0
160199,27kcZEJvhkb1rzZS9gCpdA,I've been rememberin' the mornings\nWhere we'd...,en,383,136,0.355091,151.124739,0.0
160204,0NuWgxEp51CutD2pJoF4OM,"She got the gold hoops, Prada fit\nI'm in love...",en,267,114,0.426966,153.266692,0.0


# Theme

## Filler + embed

In [14]:
import re, nltk
import numpy as np, pandas as pd
from functools import lru_cache
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

nltk.download("wordnet", quiet=True)
lemmatizer = WordNetLemmatizer()

FILLER = {"yeah","ooh","oh","uh","na","la","hey","ah","woo","whoa","mm","hmm","ha","yo",
          "gonna","wanna","gotta","cause","em","ya","ima","ayy","ay","dont","im","ive","ill",
          "youre","cant","aint","chorus","verse","intro","outro","bridge"}
STOP = ENGLISH_STOP_WORDS | FILLER

def dedupe_lines(text):
    text = re.sub(r"\[.*?\]", "", text)
    lines = [l.strip().lower() for l in text.split("\n") if l.strip()]
    return "\n".join(dict.fromkeys(lines))         

@lru_cache(maxsize=None)
def lemma(w):
    return lemmatizer.lemmatize(w)

def to_tokens(text):
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return [lemma(w) for w in text.split() if w not in STOP and len(w) > 2]

lyr = lyr.dropna(subset=["lyrics"]).reset_index(drop=True)
lyr["text_embed"] = lyr["lyrics"].apply(dedupe_lines)
lyr["text_topic"] = lyr["text_embed"].apply(lambda t: " ".join(to_tokens(t)))

lyr = lyr[lyr["text_topic"].str.split().str.len() >= 20].reset_index(drop=True)
print(len(lyr), "songs kept")

32801 songs kept


## NMF

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

K = 10
tfidf = TfidfVectorizer(max_df=0.5, min_df=5, max_features=20000)
X = tfidf.fit_transform(lyr["text_topic"])

nmf = NMF(n_components=K, init="nndsvd", random_state=42, max_iter=400)
W = nmf.fit_transform(X)                       # songs x topics
terms = np.array(tfidf.get_feature_names_out())

for k, comp in enumerate(nmf.components_):
    print(f"NMF topic {k:2d}: {', '.join(terms[comp.argsort()[::-1][:12]])}")

W_norm = W / (W.sum(axis=1, keepdims=True) + 1e-9)
lyr["nmf_topic"] = W.argmax(axis=1)
lyr["nmf_strength"] = W_norm.max(axis=1)

NMF topic  0: time, say, thing, think, tell, way, said, did, maybe, won, try, mind
NMF topic  1: nigga, bitch, fuck, shit, ain, gon, money, as, fuckin, huh, lil, hoe
NMF topic  2: love, heart, forever, true, need, fall, hold, arm, loving, kiss, way, hurt
NMF topic  3: baby, come, need, babe, crazy, tell, night, right, ain, leave, make, long
NMF topic  4: let, tonight, won, come, dance, hold, right, make, stop, ready, turn, time
NMF topic  5: want, need, tell, really, stay, say, come, hold, make, think, somebody, body
NMF topic  6: got, ain, man, little, good, boy, right, money, big, everybody, said, rock
NMF topic  7: away, day, life, night, come, light, heart, eye, world, dream, sun, sky
NMF topic  8: feel, make, way, feeling, right, good, inside, need, real, tonight, alive, alright
NMF topic  9: girl, world, boy, make, pretty, good, tell, little, look, night, man, guy


- NMF topic  0: time, say, thing, think, tell, way, said, did, maybe, won, try, mind
- NMF topic  1: nigga, bitch, fuck, shit, ain, gon, money, as, fuckin, huh, lil, hoe
- NMF topic  2: love, heart, forever, true, need, fall, hold, arm, loving, kiss, way, hurt
- NMF topic  3: baby, come, need, babe, crazy, tell, night, right, ain, leave, make, long
- NMF topic  4: let, tonight, won, come, dance, hold, right, make, stop, ready, turn, time
- NMF topic  5: want, need, tell, really, stay, say, come, hold, make, think, somebody, body
- NMF topic  6: got, ain, man, little, good, boy, right, money, big, everybody, said, rock
- NMF topic  7: away, day, life, night, come, light, heart, eye, world, dream, sun, sky
- NMF topic  8: feel, make, way, feeling, right, good, inside, need, real, tonight, alive, alright
- NMF topic  9: girl, world, boy, make, pretty, good, tell, little, look, night, man, guy

## Embed

In [27]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   

import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

#model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
model = SentenceTransformer("all-mpnet-base-v2", device=device)

embeddings = model.encode(
    lyr["text_embed"].tolist(),
    batch_size=64,                      
    show_progress_bar=True,
    normalize_embeddings=True,
)
np.save("./data/embeddings_mpnet.npy", embeddings)
print(embeddings.shape)

device: mps


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/513 [00:00<?, ?it/s]

(32801, 768)


## BERTopic

In [28]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, min_samples=10, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)
vectorizer = CountVectorizer(min_df=5, ngram_range=(1, 2))

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    top_n_words=12,
    verbose=True,
)

docs = lyr["text_topic"].tolist()
topics, _ = topic_model.fit_transform(docs, embeddings)

print("Outliers:", (np.array(topics) == -1).mean().round(3))
topic_model.get_topic_info().head(30)

2026-11-10 16:40:33,809 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-10 16:40:53,349 - BERTopic - Dimensionality - Completed ✓
2026-11-10 16:40:53,350 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-10 16:40:55,841 - BERTopic - Cluster - Completed ✓
2026-11-10 16:40:55,845 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-10 16:40:58,651 - BERTopic - Representation - Completed ✓


Outliers: 0.003


,Topic,Count,Name,Representation,Representative_Docs
0,-1,92,-1_love_don_raise_know,"[love, don, raise, know, head, baby, hear, fal...",[heard secret chord david played pleased lord ...
1,0,32417,0_like_don_know_just,"[like, don, know, just, love, got, time, let, ...",[deal shit nigga fuckin mind old shitty dick a...
2,1,150,1_love_baby_know_just,"[love, baby, know, just, like, come, want, say...",[baby love kidd baby love duniya ton wakh tain...
3,2,86,2_time_away_feel_say,"[time, away, feel, say, way, don, wow, just, h...",[feeling lonely afraid time pas away time feel...
4,3,56,3_girl_like_love_make,"[girl, like, love, make, feel, baby, hot, want...",[korean original tired sick single think need ...


<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Topic</th>
      <th>Count</th>
      <th>Name</th>
      <th>Representation</th>
      <th>Representative_Docs</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>-1</td>
      <td>92</td>
      <td>-1_love_don_raise_know</td>
      <td>[love, don, raise, know, head, baby, hear, fal...</td>
      <td>[heard secret chord david played pleased lord ...</td>
    </tr>
    <tr>
      <th>1</th>
      <td>0</td>
      <td>32417</td>
      <td>0_like_don_know_just</td>
      <td>[like, don, know, just, love, got, time, let, ...</td>
      <td>[deal shit nigga fuckin mind old shitty dick a...</td>
    </tr>
    <tr>
      <th>2</th>
      <td>1</td>
      <td>150</td>
      <td>1_love_baby_know_just</td>
      <td>[love, baby, know, just, like, come, want, say...</td>
      <td>[baby love kidd baby love duniya ton wakh tain...</td>
    </tr>
    <tr>
      <th>3</th>
      <td>2</td>
      <td>86</td>
      <td>2_time_away_feel_say</td>
      <td>[time, away, feel, say, way, don, wow, just, h...</td>
      <td>[feeling lonely afraid time pas away time feel...</td>
    </tr>
    <tr>
      <th>4</th>
      <td>3</td>
      <td>56</td>
      <td>3_girl_like_love_make</td>
      <td>[girl, like, love, make, feel, baby, hot, want...</td>
      <td>[korean original tired sick single think need ...</td>
    </tr>
  </tbody>
</table>
</div>

In [29]:
lyr["bert_topic"] = topics

In [33]:
if "nmf_topic" in lyr.columns:
    print(pd.crosstab(lyr["nmf_topic"], lyr["bert_topic"]).iloc[:, :15])

print(lyr["bert_topic"].value_counts(normalize=True).head(20).round(3))

cols = [c for c in ["artist", "title"] if c in lyr.columns]
def examples(topic, n=5):
    sub = lyr[lyr["bert_topic"] == topic]
    return sub.sample(min(n, len(sub)), random_state=1)[cols + ["lyrics"]]

examples(0)

bert_topic  -1     0   1   2   3
nmf_topic                       
0           12  4138   8  13   2
1            2  2688   9   2   1
2            8  2871  27   4   7
3            7  2127   9   3   6
4            2  1886   8   9   5
5            5  2145   7   5   8
6           10  3849  20   3  10
7           37  8097  25  29   5
8            6  2862  11  16   8
9            3  1754  26   2   4
bert_topic
 0    0.988
 1    0.005
-1    0.003
 2    0.003
 3    0.002
Name: proportion, dtype: float64


,lyrics
18727,It breaks my heart 'cause I know\nYou're the o...
710,"Cum girl, I'm tryna get your pussy wet\nWork t..."
6496,These half-truths you're telling me\nWon't bri...
14832,Here we go! (Woo!)\nC'mon!\nAlright\nLittle lo...
13818,"When I die, put my ashes in a trash bag\nI don..."


bert_topic  -1     0   1   2   3
nmf_topic                       
0           12  4138   8  13   2
1            2  2688   9   2   1
2            8  2871  27   4   7
3            7  2127   9   3   6
4            2  1886   8   9   5
5            5  2145   7   5   8
6           10  3849  20   3  10
7           37  8097  25  29   5
8            6  2862  11  16   8
9            3  1754  26   2   4
bert_topic
 0    0.988
 1    0.005
-1    0.003
 2    0.003
 3    0.002
Name: proportion, dtype: float64

In [35]:
theme = pd.read_csv('./data/theme.csv')
theme.head(20)

,id,hartman_theme,hartman_confidence,sam_theme,sam_confidence
0,5A6OHHy73AR5tLxgTc98zz,0,0.437304,27,0.725812
1,0O45fw2L5vsWpdsOdXwNAR,4,0.369275,27,0.722145
2,2zZJnAlhgvW6DERn1tu3fb,6,0.412103,18,0.358871
3,0sf12qNH5qcw8qpgymFOqD,5,0.401395,9,0.371560
4,7wZUrN8oemZfsEd1CGkbXE,5,0.523698,18,0.809899
5,4sPmO7WMQUAf45kwMOtONw,5,0.660518,24,0.573431
6,41bIQPBE1lFN0mmw6Lmssz,3,0.331537,18,0.824596
7,6fA7akEuTUL3dW1V0GELaZ,2,0.980924,14,0.831322
8,7qiZfU4dY1lWllzX7mPBI3,3,0.352158,18,0.939680
9,2VxeLyX666F8uXCJ0dZF8B,2,0.950503,14,0.501655


In [37]:
merge = pd.merge(lyr, theme, on='id')
merge.head(20)

,id,lyrics,language_ft,word_count,unique_count,TTR,yules_k,profanity_count,text_embed,text_topic,nmf_topic,nmf_strength,bert_topic,hartman_theme,hartman_confidence,sam_theme,sam_confidence
0,5A6OHHy73AR5tLxgTc98zz,"Yeah, uh-huh, you know what it is\nBlack and y...",en,656,200,0.304878,260.362509,8.0,"yeah, uh-huh, you know what it is\nblack and y...",huh know black yellow black yellow big huh scr...,1,0.415875,0,0,0.437304,27,0.725812
1,0O45fw2L5vsWpdsOdXwNAR,I'm bringin' sexy back (Yeah)\nThem other boys...,en,673,126,0.187221,347.590016,4.0,i'm bringin' sexy back (yeah)\nthem other boys...,bringin sexy boy don know act think special tu...,4,0.298611,0,4,0.369275,27,0.722145
2,2zZJnAlhgvW6DERn1tu3fb,"Ooh\nOoh, mmm, mmm\nHave you ever reached a ra...",en,389,156,0.401028,183.083439,1.0,"ooh\nooh, mmm, mmm\nhave you ever reached a ra...",mmm mmm reached rainbow end did pot gold catch...,2,0.579179,0,6,0.412103,18,0.358871
3,0sf12qNH5qcw8qpgymFOqD,Yeah\nI've been tryna call\nI've been on my ow...,en,259,115,0.444015,178.646805,0.0,yeah\ni've been tryna call\ni've been on my ow...,tryna long maybe love maybe going withdrawal d...,4,0.269882,0,5,0.401395,9,0.371560
4,7wZUrN8oemZfsEd1CGkbXE,"Closed off from love, I didn't need the pain\n...",en,379,126,0.332454,291.468528,0.0,"closed off from love, i didn't need the pain\n...",closed love didn need pain twice vain time sta...,2,0.446353,0,5,0.523698,18,0.809899
5,4sPmO7WMQUAf45kwMOtONw,"Hello, it's me\nI was wondering if after all t...",en,365,143,0.391781,137.778053,0.0,"hello, it's me\ni was wondering if after all t...",hello wondering year like meet say time suppos...,0,0.763809,0,5,0.660518,24,0.573431
6,41bIQPBE1lFN0mmw6Lmssz,New York City\nYou are now rockin'\nWith 50 Ce...,en,630,222,0.352381,224.441618,2.0,new york city\nyou are now rockin'\nwith 50 ce...,new york city rockin cent love just chill twis...,2,0.304010,0,3,0.331537,18,0.824596
7,6fA7akEuTUL3dW1V0GELaZ,And if I fall along the way\nPick me up and du...,en,290,116,0.400000,182.117195,1.0,and if i fall along the way\npick me up and du...,fall way pick dust tired make breath walk need...,8,0.329790,0,2,0.980924,14,0.831322
8,7qiZfU4dY1lWllzX7mPBI3,The club isn't the best place to find a lover\...,en,678,154,0.227139,226.103238,0.0,the club isn't the best place to find a lover\...,club isn best place lover bar friend table doi...,2,0.193141,0,3,0.352158,18,0.939680
9,2VxeLyX666F8uXCJ0dZF8B,"Tell me somethin', girl\nAre you happy in this...",en,190,97,0.510526,185.404895,0.0,"tell me somethin', girl\nare you happy in this...",tell somethin girl happy modern world need som...,9,0.323356,0,2,0.950503,14,0.501655


In [41]:
merge.shape

(32801, 17)

In [43]:
merge.to_csv('./data/merge', index=False)